In [1]:
import pandas as pd
import numpy as np

In [2]:
import langchain

In [3]:
!pip install langchain-community pypdf
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-

In [4]:
### We would load the Adobe brand guidelines PDF###
loader=PyPDFLoader("/content/Adobe_Brand_Voice.pdf")
documents=loader.load()

In [5]:
def clean_text(text):
    # Basic cleaning: remove extra spaces, line breaks
    text = text.replace("\n", " ").strip()
    return " ".join(text.split())

cleaned_texts = [clean_text(doc.page_content) for doc in documents]

In [6]:
cleaned_texts

["ADOBE Brand Voice & Tone Guidelines — For RAG Integration MISSION Changing the world through personalized digital experiences. Adobe empowers everyone — from individual creators to global enterprises — to bring their ideas to life through creativity and technology. Nurturing creativity is at the heart of everything Adobe does, for employees as well as the individuals, businesses, and communities it serves. BRAND BELIEF No matter who you are, you want to stand out. Adobe believes that creativity and technology empower everyone to reach their full potential. Whether you are an enterprise marketer, small business owner, creative professional, or just trying to create content for yourself, standing out is the key driver behind feeling successful and accomplished. VALUES Create the Future Creativity is in Adobe's DNA. We constantly look around the corner to see what is possible. We are builders, makers, and inventors driven by deep empathy for our customers. We have the courage to disrupt

In [7]:
### Chunking###
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(cleaned_texts)

In [8]:
chunks

[["ADOBE Brand Voice & Tone Guidelines — For RAG Integration MISSION Changing the world through personalized digital experiences. Adobe empowers everyone — from individual creators to global enterprises — to bring their ideas to life through creativity and technology. Nurturing creativity is at the heart of everything Adobe does, for employees as well as the individuals, businesses, and communities it serves. BRAND BELIEF No matter who you are, you want to stand out. Adobe believes that creativity and technology empower everyone to reach their full potential. Whether you are an enterprise marketer, small business owner, creative professional, or just trying to create content for yourself, standing out is the key driver behind feeling successful and accomplished. VALUES Create the Future Creativity is in Adobe's DNA. We constantly look around the corner to see what is possible. We are builders, makers, and inventors driven by deep empathy for our customers. We have the courage to disrup

In [29]:
!pip install -U sentence-transformers
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Flatten the chunks list into a single list of strings
flat_chunks = [item for sublist in chunks for item in sublist]

doc_embeddings = embedder.encode(flat_chunks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [33]:
import faiss

### Storing the content in FAISS###
index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(doc_embeddings)

In [36]:
!pip install faiss-cpu

In [40]:
### Querying the Guidelines###
query = "What is Adobe' brand beleif?"
query_embedding = embedder.encode([query])
D, I = index.search(query_embedding, k=3)  # retrieve top 3 chunks
retrieved_docs = [flat_chunks[i] for i in I[0]]

In [41]:
print(retrieved_docs)

["ADOBE Brand Voice & Tone Guidelines — For RAG Integration MISSION Changing the world through personalized digital experiences. Adobe empowers everyone — from individual creators to global enterprises — to bring their ideas to life through creativity and technology. Nurturing creativity is at the heart of everything Adobe does, for employees as well as the individuals, businesses, and communities it serves. BRAND BELIEF No matter who you are, you want to stand out. Adobe believes that creativity and technology empower everyone to reach their full potential. Whether you are an enterprise marketer, small business owner, creative professional, or just trying to create content for yourself, standing out is the key driver behind feeling successful and accomplished. VALUES Create the Future Creativity is in Adobe's DNA. We constantly look around the corner to see what is possible. We are builders, makers, and inventors driven by deep empathy for our customers. We have the courage to disrupt